# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [2]:
import warnings
from os.path import join

import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display, Markdown, display_html

In [3]:
%autoreload
from datasim.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity"

In [5]:
adata_query = sc.read_h5ad(join(DATA_PATH, "01ad3cd7-3929-4654-84c0-6db05bd5fd59_processed.h5ad"))
adata_ref = sc.read_h5ad(join(DATA_PATH, "b0e547f0-462b-4f81-b31b-5b0a5d96f537_processed.h5ad"))

In [6]:
adata_query.var.set_index("gene_names", inplace=True)
adata_ref.var.set_index("gene_names", inplace=True)

In [7]:
sc.pp.normalize_total(adata_query)
sc.pp.log1p(adata_query)
sc.pp.highly_variable_genes(adata_query, n_top_genes=2000, subset=True)

sc.pp.normalize_total(adata_ref)
sc.pp.log1p(adata_ref)
sc.pp.highly_variable_genes(adata_ref, n_top_genes=2000, subset=True)

/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]
/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


In [8]:
adata_query

AnnData object with n_obs × n_vars = 600929 × 2000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [9]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 2000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [10]:
cluster_mapping = pd.read_parquet(join(DATA_PATH, "model-output/cluster_mapping.parquet"))
cluster_distance = pd.read_parquet(join(DATA_PATH, "model-output/cluster_distance.parquet"))

In [11]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [12]:
top_n_labels = DatasetMapping.select_most_similar_clusters(cluster_mapping, cluster_distance, n_top=4)

#### Author provided cluster labels

In [13]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{k}**: {v}"))

*1*: **B_Mem**: ['IGHMhi_memory_B', 'B', 'IGHMlo_memory_B']

*2*: **B_Mem_Prolif_Early**: ['IGHMlo_memory_B', 'IGHMhi_memory_B']

*3*: **B_Mem_Prolif_Late**: ['IGHMlo_memory_B', 'IGHMhi_memory_B']

*4*: **B_Naive**: ['naive_B']

*5*: **B_Naive_Pool3**: []

...

*36*: **cM_Act_1006**: []

*37*: **cM_IFN_1006**: []

*38*: **ncM**: ['CD16+_Monocyte']

*39*: **ncM_1006**: ['CD16+_Monocyte']

*40*: **pDC**: ['pDC', 'DC']

#### Ontology mapped cluster labels

In [14]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]}"))

*1*: **B cell**: ['memory B cell', 'B cell', 'memory B cell']

*2*: **B cell**: ['memory B cell', 'memory B cell']

*3*: **B cell**: ['memory B cell', 'memory B cell']

*4*: **B cell**: ['naive B cell']

*5*: **B cell**: []

...

*36*: **classical monocyte**: []

*37*: **classical monocyte**: []

*38*: **non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

*39*: **non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

*40*: **plasmacytoid dendritic cell**: ['plasmacytoid dendritic cell', 'dendritic cell']

## Evaluate cluster similarity

In [15]:
%autoreload
from datasim.utils import get_differentially_expressed_genes

In [16]:
def highly_expressed_genes(adata, n_genes):
    highly_expressed = {}
    
    for cluster in adata.obs["cell_type_author"].unique():
        avg_expression = np.array(
            adata[adata.obs["cell_type_author"] == cluster].X.mean(axis=0)
        ).flatten()
        
        highly_expressed_genes_idxs = np.argsort(-avg_expression)[:n_genes]
        
        highly_expressed[cluster] = {
            "gene": adata.var.index[highly_expressed_genes_idxs].tolist(),
            "average_expression": avg_expression[highly_expressed_genes_idxs]
        }

    return highly_expressed


In [ ]:
METHOD = "wilcoxon"
P_VAL_THRESHOLD = 0.01

# ignore warnings here as scanpy.tl.rank_genes_groups throws a lot of warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    de_genes_query = get_differentially_expressed_genes(
        adata_query, 
        "cell_type_author", 
        n_genes=15,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )
    de_genes_ref = get_differentially_expressed_genes(
        adata_ref, 
        "cell_type_author", 
        n_genes=15,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )


In [ ]:
highly_expressed_query = highly_expressed_genes(adata_query, n_genes=100)
highly_expressed_ref = highly_expressed_genes(adata_ref, n_genes=100)

In [ ]:
N_GENES_TO_SHOW = 15


for k, v in top_n_labels.items():
    html = [f"<h1 id='{k}'> <b>{k}</b>: {v} </h1>"]

    if v:
        # Add summary for differentially expressed genes
        html.append("<b><i>Overlap of DE genes:</i></b> <br /><br />")

        def style_overlap(v, props=""):
            top_n_query_genes = de_genes_query[k]["gene"][:N_GENES_TO_SHOW].tolist()
            return "color:green;" if v in top_n_query_genes else "color:red;"

        html += [
            de_genes_query[k]
            .head(N_GENES_TO_SHOW)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html += [
            de_genes_ref[gene]
            .head(N_GENES_TO_SHOW)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
        html.append("<br /><br />")

        # Add summary for highly expressed genes
        html.append("<b><i>Overlap of highly expressed genes:</i></b> <br /><br />")

        def style_overlap(v, propgs=""):
            top_n_query_genes = highly_expressed_query[k]["gene"][:N_GENES_TO_SHOW]
            return "color:green;" if v in top_n_query_genes else "color:red;"

        html += [
            pd.DataFrame(highly_expressed_query[k])
            .head(N_GENES_TO_SHOW)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html += [
            pd.DataFrame(highly_expressed_ref[gene])
            .head(N_GENES_TO_SHOW)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
    else:
        html.append("<b><i>No matches found</i></b>")

    display_html("".join(html), raw=True)
    with open(join("cluster-evaluation-output", f"{k}.html"), "w") as f:
        f.write("".join(html))

    print("\n")
